In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

# Read the dataset Q1_data.csv using read_csv()
import os
import pandas as pd

q1 = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(q1)


In [ ]:
# Task 2: Write your code here:

# Inspect the first few rows using head()
df.head()

In [ ]:
# Task 3: Write your code here:

# Display dataset information using info()
df.info()

In [ ]:
# Task 4: Write your code here:

# Show statistical description using describe()
df.describe()

In [ ]:
# Task 5: Write your code here:

# Plot the target distribution (delivery_time)
import matplotlib.pyplot as plt

df['Delivery_Time'].hist()
plt.xlabel('Delivery Time in minutes')
plt.ylabel('frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:

# Drop the 'Order_ID' column from the data
df = df.drop('Order_ID', axis=1)
df

In [ ]:
# Task 2: Write your code here:

# Handle missing values appropriately
# (Hint: I guess you want to have a closer look at the columns with missing values :) )

print('missing values count:',)
print(df.isna().sum())

catag_feat = df.select_dtypes('object').columns
for col in catag_feat:
  df[col] = df[col].fillna(df[col].mode()[0])

num_feat = df.select_dtypes('number').columns
for col in num_feat:
  df[col] = df[col].fillna(df[col].mean()) # median will work too


print('missing values count:',)
print(df.isna().sum())


In [ ]:
# Task 3: Write your code here:

# Check and remove duplicates if any exist
if (df.duplicated().sum() > 0):
  df = df.drop_duplicates()

In [ ]:
# Task 4: Write your code here:

# Encode categorical variables if needed (Bonus if used One Hot Encoding)
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

for col in catag_feat:
  print(col, df[col].unique())
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])

# ohe.fit_transform(pd.DataFrame(df['Weather']),)
df

In [ ]:
# Task 5: Write your code here:

# Apply feature scaling for all features (Use StandardScaler)
from sklearn.preprocessing import StandardScaler
ss = StandardScaler()

# don't scale the target
to_scale = df.drop('Delivery_Time', axis=1).columns
df[to_scale] = ss.fit_transform(df[to_scale])

df

In [ ]:
# Task 6: Write your code here:

# Check for target imbalance and state if it is imbalanced or not
# (keep this cell empty if not needed)

# target imbalance is a concern in **regression** problems as well,
# we deal with it with (log transformations)
# but I'm not sure how to implement it :P
df['Delivery_Time'].hist()
plt.show()
# our data is a bit skewed

In [ ]:
# Task 1: Write your code here:

# Split the dataset into features (X) and target (y)
X = df.drop('Delivery_Time', axis=1)
y = df['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
import numpy as np
# Use the correct split: KFold OR StratifiedKFold
from sklearn.model_selection import KFold
kf = KFold(n_splits=5)

# Train a RandomForest model
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)

# Evaluate using MAE (Mean Absolute Error) ONLY
from sklearn.metrics import mean_absolute_error
scores_mae = []
last_pred = []
# 5-Fold cross-validation
for train_index, test_index in kf.split(X, y):
    # Split data into training and testing sets
    X_Train, X_Test = X.iloc[train_index,:], X.iloc[test_index,:]
    y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]
    # Train the model
    model.fit(X_Train, y_Train)
    # Predict on the test set
    y_pred = model.predict(X_Test)
    # Calculate metrics
    scores_mae.append(mean_absolute_error(y_Test, y_pred))
    last_pred = y_pred

# Print the averaged score across all folds
print(np.sum(scores_mae) / len(scores_mae))

In [ ]:
# Task 1: Write your code here:

# Feature importance
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='blue')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

# Plot predicted delivery time histogram

In [ ]:
# Task Bonus: Write your code here: